# 🌿 Plant Doctor – Huấn luyện trực tiếp trên Google Drive

**Runtime → Change runtime type → GPU (T4)**.

Toàn bộ code, dữ liệu ảnh và các mô hình sau huấn luyện sẽ được **đọc và lưu trực tiếp trên Google Drive** (`MyDrive/plant_doctor/`). Không cần copy hay đồng bộ thủ công, nếu Colab bị ngắt kết nối thì mô hình vẫn còn nguyên vẹn trên Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Chuyển trực tiếp vào thư mục dự án trên Google Drive
%cd /content/drive/MyDrive/plant_doctor
!pwd
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt --no-warn-conflicts

## Bước 0 – Chuyển dataset phân loại → YOLO detection (70/15/15, seed 42)
Đọc ảnh trực tiếp từ `data/raw/` trên Drive và tạo dataset chuẩn YOLO vào `data/yolo/` trên Drive.
`--max-per-class 400` để thử nhanh; bỏ tham số này để dùng toàn bộ ảnh.

In [ ]:
!python -m plant_doctor.data.prepare_dataset --src data/raw --method opencv --max-per-class 400

## Bước 1 – Tự huấn luyện YOLOv8n
Mô hình tốt nhất (`best.pt`) sẽ tự động lưu thẳng vào Drive: `models/yolo_leaf_disease/weights/best.pt`

In [ ]:
!python -m plant_doctor.training.train_yolo --epochs 100 --batch 16

## Bước 2 – Tinh chỉnh mô hình Embedding
Lưu trực tiếp vào Drive: `models/custom_embedding/`

In [ ]:
!python -m plant_doctor.training.train_embedding --epochs 20

## Bước 3 – Fine-tune Qwen2.5-1.5B-Instruct bằng LoRA (r=16)
Mô hình đã gộp LoRA sẽ lưu thẳng vào Drive: `models/custom_llm_lora/`

In [ ]:
!python -m plant_doctor.training.train_llm --epochs 3

## Bước 4 – Xây dựng kho FAISS cục bộ
Lưu trực tiếp vào Drive: `vector_store/faiss_index/`

In [ ]:
!python -m plant_doctor.rag.build_index

## Dọn dẹp checkpoint tạm & Kiểm tra kết quả trên Drive

In [ ]:
# Xóa thư mục checkpoint LoRA trung gian để tiết kiệm dung lượng Drive
!rm -rf models/llm_lora_checkpoints
!du -sh models/* vector_store

## Bước 5 – Chạy thử app Streamlit trên Colab (Tùy chọn)
Colab không mở được cổng trực tiếp nên dùng đường hầm Cloudflare **chỉ để demo**.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run app/streamlit_app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 8 && ./cloudflared tunnel --url http://localhost:8501